In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import argrelextrema
%matplotlib inline
import torch
from torch import nn
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import matplotlib as mpl
import matplotlib.ticker as mticker

In [ ]:
all_data = pd.read_csv('trainHH_data.csv')
num_vehicles = all_data['Vehicle_ID'].nunique()
selected_vehicles = all_data['Vehicle_ID'].unique()
specific_selected_vehicles = selected_vehicles[3001:6000]  
all_data = all_data[all_data['Vehicle_ID'].isin(specific_selected_vehicles)]

In [ ]:
class Segmentation():
    def __init__(self,all_data,desired_vehicle_id):
        self.all_data = all_data
        self.unique_vehicle_ids = all_data['Vehicle_ID'].unique()
        self.desired_vehicle_id = desired_vehicle_id

    def cal_inflection_points(self,data):
        local_minima = argrelextrema(data, np.less)
        local_maxima = argrelextrema(data, np.greater)
        inflection_points = np.sort(np.concatenate((local_minima[0], local_maxima[0])))
        inflection_indexs = [i for i in inflection_points]
        inflection_indexs.insert(0,0)
        inflection_indexs.append(len(data)-1)
        inflection_coordinates = [data[i] for i in inflection_indexs]
        delta_x = np.diff(inflection_indexs)
        delta_y = np.diff(inflection_coordinates)
        delta_x = np.append(delta_x, 0)
        delta_y = np.append(delta_y, 0)

        return inflection_indexs, inflection_coordinates,delta_x,delta_y

    def generate_result_dataset(self,variables): #variables: Velocity_x, Acceleration_x, distance, delta_v
        results = []
        for vehicle_id in self.unique_vehicle_ids:
            vehicle_data = all_data[all_data['Vehicle_ID'] == vehicle_id]
            time_id = vehicle_data['FrameID'].values
            speed = vehicle_data[str(variables)].values
            inflection_points, inflection_coordinates, delta_x, delta_y = self.cal_inflection_points(speed)
            inflection_time_ids = [time_id[i] for i in inflection_points]
            result_data = pd.DataFrame({
                'VehicleID': vehicle_id,
                'InflectionTimeID': inflection_time_ids,
                'InflectionSpeed': inflection_coordinates,
                'DeltaX': delta_x,
                'DeltaY': delta_y
            })
            results.append(result_data)
        result_dataset = pd.concat(results, ignore_index=True)
        return result_dataset

    def find_data(self,result_dataset):
        single_vehicle_result = result_dataset[result_dataset['VehicleID'] == self.desired_vehicle_id]
        single_vehicle_data = all_data[all_data['Vehicle_ID'] == self.desired_vehicle_id]
        delta_t = single_vehicle_result['DeltaX'].values
        delta_y = single_vehicle_result['DeltaY'].values
        time_turning_points = single_vehicle_result['InflectionTimeID'].values
        turning_points_values = single_vehicle_result['InflectionSpeed'].values
        return single_vehicle_data,delta_t,delta_y,time_turning_points,turning_points_values

    def get_states_set(self,delta_y,theta1,theta2):
        states = []
        # try 1 st
        for dy in delta_y:
            if dy >= theta1:
                state = 'I'
            elif dy <= theta2:
                state = 'D'
            else:
                state = 'K'
            states.append(state)
        return states

    def merge(self,states,delta_t,delta_y,time_turning_points,theta1,theta2,gama):
        merged_index_set = []
        for index, j in enumerate(states):
            if index < 1:
                if states[index+1] != j:
                    if delta_t[index] <= gama and theta2 <= delta_y[index] <= theta1:
                        states[index] = states[index-1]
                        merged_index_set.append(index)
            elif index == len(states)-1:
                if states[index-1] != j:
                    if delta_t[index] <= gama and theta2 <= delta_y[index] <= theta1:
                        states[index] = states[index-1]
                        merged_index_set.append(index)
            else:
                if states[index-1] != j and states[index+1] != j:
                    if delta_t[index] <= gama and theta2 <= delta_y[index] <= theta1:
                        states[index] = states[index-1]
                        merged_index_set.append(index)
                elif states[index-1] == j and states[index+1] != j:
                    if delta_t[index] <= gama and theta2 <= delta_y[index] <= theta1:
                        states[index] = states[index-1]
                        merged_index_set.append(index)
                elif states[index+1] == j and states[index-1] != j:
                    if delta_t[index] <= gama and theta2 <= delta_y[index] <= theta1:
                        states[index] = states[index+1]
                        merged_index_set.append(index)
        try:
            merged_index_set.remove(0)
        except:
            pass
        try:
            merged_index_set.remove(len(time_turning_points)-1)
        except:
            pass
        time_turning_points_final = np.delete(time_turning_points, merged_index_set)

        return time_turning_points_final,merged_index_set

    def cal_states_final(self,states,variable,turning_point,merged_index_set):
        if str(variable) == 'a_follow':
            variable_threshold = 0.25 # 0.2
        elif str(variable) == 'smooth_v_follow':
            variable_threshold = 13
        elif str(variable) == 'time_headway':
            variable_threshold = 1.5  # distance, 10, 12
        elif str(variable) == 'smooth_delta_v':
            variable_threshold = 0.8
        else:
            variable_threshold = np.inf
        for index, point in enumerate(turning_point):
            if states[index] == 'K':
                if index != turning_point.shape[0]-1:
                    next_point = turning_point[index]
                else:
                    next_point = turning_point[-1]
                point -= turning_point[0]
                next_point -= turning_point[0]
                mean_variable = np.mean(single_vehicle_data[str(variable)][point:next_point+1])
                if mean_variable >= variable_threshold or mean_variable <= -variable_threshold:
                    states[index] = 'K'
                else:
                    states[index] = 'K'
        states_final = [i for index,i in enumerate(states) if index not in merged_index_set]
        return states_final

    def total_turning_points(self,v_turning_points,a_turning_points,d_turning_points,dv_turning_points,gamma):
        total_ = np.concatenate((v_turning_points,a_turning_points,d_turning_points,dv_turning_points))
        total_ = np.unique(total_)
        total_points = []
        real_points = []
        for i, num in enumerate(total_):
            if i == 0:
                total_points.append(num)
            else:
                v_close = self.find_nearest_smaller(v_turning_points,num)
                a_close = self.find_nearest_smaller(a_turning_points,num)
                d_close = self.find_nearest_smaller(d_turning_points,num)
                dv_close = self.find_nearest_smaller(dv_turning_points,num)
                if (num-total_points[-1]) >=gamma:
                    if (num-v_close)>=gamma and (num-a_close)>=gamma and (num-d_close)>=gamma and (num-dv_close)>=gamma:
                        total_points.append(total_[i-1])
                        total_points.append(num)
                        real_points.append(total_[i-1])
        if total_[-1] not in total_points:
            total_points.append(total_[-1])
            real_points.append(total_[-1])
        total_points = sorted(set(total_points))
        real_points = sorted(set(real_points))
        return total_points,real_points

    def total_turning_points_without_gamma(self,v_turning_points,a_turning_points,d_turning_points,dv_turning_points):
        total_ = np.concatenate((v_turning_points,a_turning_points,d_turning_points,dv_turning_points))
        total_ = np.unique(total_)
        # total_points = sorted(set(total_))
        return total_

    def find_nearest_smaller(self,array, value):
        array = np.array(array)
        smaller_values = array[array < value]
        if smaller_values.size == 0:
            return value
        nearest_index = np.abs(smaller_values - value).argmin()
        return smaller_values[nearest_index]

    def find_nearest_smaller2(self,array, value):
        array = np.array(array)
        smaller_values = array[array <= value]
        if smaller_values.size == 0:
            return None
        nearest_index = np.abs(smaller_values - value).argmin()
        return smaller_values[nearest_index]

    def four_variable_point_state(self,four_variable_turning_points,time_turning_points,states_final):
        four_state = list(states_final)
        four_point = list(time_turning_points)
        for i, total_point in reversed(list(enumerate(four_variable_turning_points))):
            for j, base_point in enumerate(time_turning_points):
                if j != time_turning_points.shape[0]-1:
                    if total_point > base_point and total_point < time_turning_points[j+1]:
                        four_point.insert(j+1,total_point)
                        four_state.insert(j+1,states_final[j])
                    break
        return four_point,four_state

    def merge_adj_same_states(self,final_points,final_states):
        del_index_set = []
        for index, point in enumerate(final_points):
            if index != 0 and index != len(final_points)-1:
                if final_states[index] == final_states[index-1]:
                    del_index_set.append(index)
        final_points = [item for index, item in enumerate(final_points) if index not in del_index_set]
        final_states = [item for index, item in enumerate(final_states) if index not in del_index_set]
        return final_points,final_states

    def find_four_states(self,four_df):
        four_states =[]
        v_final_point4 = four_df['velocity_turning_points']
        v_final_states4 = four_df['velocity_states']
        a_final_point4 = four_df['acceleration_turning_points']
        a_final_states4 = four_df['acceleration_states']
        d_final_point4 = four_df['distance_turning_points']
        d_final_states4 = four_df['distance_states']
        dv_final_point4 = four_df['delta_v_turning_points']
        dv_final_states4 = four_df['delta_v_states']
        four_variable_turning_points = four_df['four_turning_points']
        four_real_points = four_df['four_real_points']
        for i in four_real_points:
            v_loc = v_final_point4.index(self.find_nearest_smaller2(v_final_point4,i))
            a_loc = a_final_point4.index(self.find_nearest_smaller2(a_final_point4,i))
            d_loc = d_final_point4.index(self.find_nearest_smaller2(d_final_point4,i))
            dv_loc = dv_final_point4.index(self.find_nearest_smaller2(dv_final_point4,i))
            four_states.append([v_final_states4[v_loc],a_final_states4[a_loc],d_final_states4[d_loc],dv_final_states4[dv_loc]])
        return four_states


    def all_states(self,four_variable_turning_points_wo_gama,final_point4,final_states4):
        states = []
        for i, value_i in enumerate(four_variable_turning_points_wo_gama):
            for j,value_j in enumerate(final_point4):
                if j != len(final_point4)-1:
                # print(value_i,final_point4,'wqwqwqwqw')
                    if value_i == value_j:
                        states.append(final_states4[j])
                    elif value_i > value_j and value_i < final_point4[j+1]:
                        states.append(final_states4[j])
                    else:
                        continue
                else:
                    states.append(final_states4[j])
                    break
        return states

    def binary_states(self,four_variable_turning_points_wo_gama,v_all_state4,a_all_state4,d_all_state4,dv_all_state4,gamma):
        binary_state = []
        binary_state1 = []
        binary_state2 = []
        for i, value in enumerate(four_variable_turning_points_wo_gama):
            if i != len(four_variable_turning_points_wo_gama)-1:

                if four_variable_turning_points_wo_gama[i+1]-value <gamma:
                    binary_state.append((v_all_state4[i],a_all_state4[i],d_all_state4[i],dv_all_state4[i],'short'))
                    binary_state1.append((v_all_state4[i],a_all_state4[i],d_all_state4[i],dv_all_state4[i]))
                    binary_state2.append('short')
                else:
                    binary_state.append((v_all_state4[i],a_all_state4[i],d_all_state4[i],dv_all_state4[i],'long'))
                    binary_state1.append((v_all_state4[i],a_all_state4[i],d_all_state4[i],dv_all_state4[i]))
                    binary_state2.append('long')

        return binary_state,binary_state1,binary_state2

    def binary_states_del_too_short(self,four_variable_turning_points_wo_gama,v_all_state4,a_all_state4,d_all_state4,dv_all_state4,gamma):
        binary_state_not_short = []
        binary_state1_not_short = []
        binary_state2_not_short = []
        too_short = 10
        four_variable_turning_points_wo_gama_del_too_short = []
        for i, value in enumerate(four_variable_turning_points_wo_gama):
            if i != len(four_variable_turning_points_wo_gama)-1:

                if four_variable_turning_points_wo_gama[i+1]-value < gamma:
                    if four_variable_turning_points_wo_gama[i+1]-value >= too_short:
                        binary_state_not_short.append(((v_all_state4[i],a_all_state4[i],d_all_state4[i],dv_all_state4[i]),'st'))
                        binary_state1_not_short.append((v_all_state4[i],a_all_state4[i],d_all_state4[i],dv_all_state4[i]))
                        binary_state2_not_short.append('st')
                        four_variable_turning_points_wo_gama_del_too_short.append(value)
                else:
                    binary_state_not_short.append(((v_all_state4[i],a_all_state4[i],d_all_state4[i],dv_all_state4[i]),'lg'))
                    binary_state1_not_short.append((v_all_state4[i],a_all_state4[i],d_all_state4[i],dv_all_state4[i]))
                    binary_state2_not_short.append('lg')
                    four_variable_turning_points_wo_gama_del_too_short.append(value)
        return four_variable_turning_points_wo_gama_del_too_short,binary_state_not_short,binary_state1_not_short,binary_state2_not_short

    def choose_action_phase(self, single_four_points, single_vehicle_data,binary_state1):
        action_phase = []
        too_short = 10
        four_variable_action_point = []
        labels = []
        for i, value in enumerate(single_four_points):
            if i != len(single_four_points)-1:
                if single_four_points[i+1]-value >= too_short:
                    index = np.where(single_vehicle_data['FrameID'].values == value)[0][0]
                    index_next = np.where(single_vehicle_data['FrameID'].values == single_four_points[i+1])[0][0]
                    if index_next-index >= too_short:
                        action_phase.append((single_vehicle_data['smooth_v_follow'].values[index: index_next+1],single_vehicle_data['a_follow'].values[index: index_next+1],single_vehicle_data['time_headway'].values[index: index_next+1],single_vehicle_data['smooth_delta_v'].values[index: index_next+1]))
                        four_variable_action_point.append(value)
                        labels.append(binary_state1[i])
        return four_variable_action_point, action_phase,labels

    def plt_segment(self,variable,y_label,single_vehicle_data,state_final,turning_point,time_turning_points_og):
        # create a figure and axis
        plt.figure(figsize=(12, 2))
        x_min = min(single_vehicle_data['FrameID'])
        x_max = max(single_vehicle_data['FrameID'])
        plt.xlim(x_min, x_max)
        plt.plot(single_vehicle_data['FrameID'].values, single_vehicle_data[str(variable)].values, color = 'red', linestyle='-', linewidth=1)

        plt.xlabel('Time')
        plt.ylabel(str(y_label))
        plt.title('Segmentation')

        # add vertical lines for turning points
        for i, state in enumerate(time_turning_points_og):
            start = time_turning_points_og[i]
            if start == time_turning_points_og[-1]:
                end = time_turning_points_og[i]
            else:
                end = time_turning_points_og[i+1]
            x_pos = (start + end) / 2
            plt.axvline(x=start, color='grey', linestyle='--', alpha=0.5)
        plt.axvline(x=time_turning_points_og[-1], color='grey', linestyle='--', alpha=0.5)
        plt.axvline(x=0, color='grey', linestyle='--', alpha=0.5)

        # add colored regions for states
        for i in range(len(state_final)):
            start = turning_point[i]
            if i != len(state_final)-1:
                end = turning_point[i + 1]
            else:
                end = int(single_vehicle_data['FrameID'].values[-1])

            color = None
            if state_final[i] == 'I':
                color = 'red'
            elif state_final[i] == 'D':
                color = 'yellow'
            elif state_final[i] == 'K':
                color = 'green'
            # elif state_final[i] == 'D':
            #     color = 'blue'
            plt.axvspan(start, end, color=color, alpha=0.4,lw=0)
        plt.show()
        return

    def plt_rule_label(self,single_vehicle_data,state_final,turning_point):
        # plt.figure(figsize=(10, 5))
        fig, ax = plt.subplots(figsize=(12, 2))
        ax2 = ax.twinx()
        x_min = min(single_vehicle_data['FrameID'])
        x_max = max(single_vehicle_data['FrameID'])
        plt.xlim(x_min, x_max)
        variable = ['smooth_v_follow','a_follow','time_headway','smooth_delta_v']
        color_set = ['red','green','blue','yellow']
        y_label = ['$v$','$a$','$T$','$\Delta v$']
        for i in range(len(variable)):
            if i !=0:
                ax2.plot(single_vehicle_data['FrameID'].values, single_vehicle_data[str(variable[i])].values, color = color_set[i], linestyle='-', linewidth=1,label = str(y_label[i]))
            else:
                ax.plot(single_vehicle_data['FrameID'].values, single_vehicle_data[str(variable[i])].values, color = color_set[i], linestyle='-', linewidth=1,label = str(y_label[i]))
        ax.set_xlabel('Time')

        plt.title('Segmentation_rule_based')

    def plt_rule_label(self, single_vehicle_data, state_final, turning_point):
        mpl.rcParams['font.family'] = 'Arial'  # Example: 'serif', 'sans-serif', 'Arial', etc.
        font_settings = {
        'title': {'fontsize': 18, 'fontweight': 'normal'},
        'label': {'fontsize': 18, 'fontweight': 'normal'},
        'tick': {'fontsize': 18, 'fontweight': 'normal'}}
        tick_fontsize = 18  # You can adjust this value as needed
        legend_fontsize = 20  # Font size for the legend

        fig, ax = plt.subplots(figsize=(8, 4))
        ax2 = ax.twinx()
        x_min = min(single_vehicle_data['FrameID'])
        x_max = max(single_vehicle_data['FrameID'])
        plt.xlim(x_min, x_max)

        variable = ['smooth_v_follow', 'a_follow', 'time_headway', 'smooth_delta_v']
        color_set = ['#DF826C', '#3081D0', '#65B741', '#C683D7']
        y_label = ['$v$', '$a$', '$T$', '$\Delta v$']
        line_styles = ['-', '--', '-.', ':']  # Different line styles
        for i in range(len(variable)):
            if i != 0:
                ax2.plot(single_vehicle_data['FrameID'].values, single_vehicle_data[str(variable[i])].values, color=color_set[i], linestyle=line_styles[i], linewidth=1.5, label=y_label[i])
            else:
                ax.plot(single_vehicle_data['FrameID'].values, single_vehicle_data[str(variable[i])].values, color=color_set[i], linestyle=line_styles[i], linewidth=1.5, label=y_label[i])

        ax.set_xlabel('Timestamp(0.1s)', **font_settings['label'])
        ax.set_ylabel('$v$(m/s)', **font_settings['label'])
        ax2.set_ylabel('$a$(m/s2) / $T$(s) / $\Delta v$(m/s)', **font_settings['label'])

        # plt.title('Segmentation_rule_based', **font_settings['title'])
        ax.tick_params(axis='both', which='major', labelsize=tick_fontsize)
        ax2.tick_params(axis='both', which='major', labelsize=tick_fontsize)
        y_formatter = mticker.FuncFormatter(lambda x, pos: f"{int(x)}")
        ax.yaxis.set_major_formatter(y_formatter)
        ax2.yaxis.set_major_formatter(y_formatter)

        # If you need legends, you can uncomment these lines
        # ax.legend(loc='upper left',fontsize=legend_fontsize)
        # ax2.legend(loc='upper right',fontsize=legend_fontsize)
        lines = []  # To store the line objects for the legend
        for i in range(len(variable)):
            line, = ax2.plot(single_vehicle_data['FrameID'].values, single_vehicle_data[str(variable[i])].values, color=color_set[i], linestyle=line_styles[i], linewidth=1.5, label=y_label[i]) if i != 0 else ax.plot(single_vehicle_data['FrameID'].values, single_vehicle_data[str(variable[i])].values, color=color_set[i], linestyle=line_styles[i], linewidth=1.5, label=y_label[i])
            lines.append(line)
        fig.legend(handles=lines, loc='upper right', bbox_to_anchor=(1, 1.13), ncol=len(lines), fontsize=legend_fontsize, frameon=False)
        plt.tight_layout()

        # fill color for states
        for i in range(len(state_final)):
            start = turning_point[i]
            if i != len(state_final)-1:
                end = turning_point[i + 1]
            else:
                end = int(single_vehicle_data['FrameID'].values[-1])

            color = None
            if state_final[i] == 'Speed_up':
                color = '#DF826C'  # red
            elif state_final[i] == 'Slow_down':
                color = '#F8FFD2' # light yellow
            elif state_final[i] == 'Catch_up':
                color = '#C1F2B0'   # green
            elif state_final[i] == 'Fall_behind':
                color = '#B4BDFF'  # purple
            elif state_final[i] == 'Follow_behind':
                color = '#FFE3BB'  # yellow
            elif state_final[i] == 'Hold_speed':
                color = '#6DB9EF'  # blue
            # elif state_final[i] == 'D':
            #     color = 'blue'
            ax.axvspan(start, end, color=color, alpha=0.6,lw=0)
        plt.show()
        return


    def rule_label(self,states_v_final,turning_points_final_v,states_a_final,turning_points_final_a,states_d_final,turning_points_final_d):
        rule_based_label = []
        rule_based_points = []
        for v_index, state_v in enumerate(states_v_final):
            if state_v == 'I':
                rule_based_label.append('Speed_up')
                rule_based_points.append(turning_points_final_v[v_index])
            elif state_v == 'D':
                rule_based_label.append('Slow_down')
                rule_based_points.append(turning_points_final_v[v_index])
            else:
                turning_point_v = turning_points_final_v[v_index]
                d_ = turning_points_final_d[turning_points_final_d <= turning_point_v].max() if np.any(turning_points_final_d <= turning_point_v) else None
                index_d_ = np.where(turning_points_final_d == d_)[0][0] if d_ is not None else None
                if turning_points_final_d[index_d_] in rule_based_points:
                    continue
                if len(rule_based_points) != 0:
                    if turning_points_final_d[index_d_] - rule_based_points[-1] < 10:
                        continue
                if states_d_final[index_d_] == 'I':
                    rule_based_label.append('Fall_behind')
                    rule_based_points.append(turning_points_final_d[index_d_])
                elif states_d_final[index_d_] == 'D':
                    rule_based_label.append('Catch_up') 
                    rule_based_points.append(turning_points_final_d[index_d_])
                else:
                    turning_point_d = turning_points_final_d[index_d_]
                    a_ = turning_points_final_a[turning_points_final_a <= turning_point_d].max() if np.any(turning_points_final_a <= turning_point_d) else None
                    index_a_ = np.where(turning_points_final_a == a_)[0][0] if a_ is not None else None
                    if turning_points_final_a[index_a_] in rule_based_points:
                        continue
                    if len(rule_based_points) != 0:
                        if turning_points_final_a[index_a_] - rule_based_points[-1] < 10:
                            continue
                    if states_a_final[index_a_] == 'I':
                        rule_based_label.append('Follow_behind')
                        rule_based_points.append(turning_points_final_a[index_a_])
                    elif states_a_final[index_a_] == 'D':
                        rule_based_label.append('Follow_behind')
                        rule_based_points.append(turning_points_final_a[index_a_])
                    else:
                        rule_based_label.append('Hold_speed')
                        rule_based_points.append(turning_points_final_a[index_a_])
        return rule_based_label,rule_based_points




    def fill_color(self,velocity, acceleration, distance, delta_v, dataframe,axvspan,four_variable_turning_points,single_vehicle_data):
        COLOR_SET = [['#AED6F1', '#85C1E9', '#5DADE2', '#3498DB'],['#A9DFBF', '#82E0AA', '#58D68D', '#2ECC71'],['#F5B7B1', '#F1948A', '#EC7063', '#E74C3C'],['#D7BDE2', '#C39BD3', '#AF7AC5', '#9B59B6']]
        loc= [3.5,2.5,1.5,.5]

        fig,ax = plt.subplots(1,1,figsize=(20,1.5))
        plt.ylim(0,4)
        ax.yaxis.set_major_locator(plt.MultipleLocator(1))
        velocity_turning_points_final = dataframe['velocity_turning_points']
        velocity_states_final = dataframe['velocity_states']
        acceleration_turning_points_final =  dataframe['acceleration_turning_points']
        acceleration_states_final = dataframe['acceleration_states']
        distance_turning_points_final =  dataframe['distance_turning_points']
        distance_states_final = dataframe['distance_states']
        delta_v_turning_points_final =  dataframe['delta_v_turning_points']
        delta_v_states_final = dataframe['delta_v_states']
        if velocity:
            color_set = COLOR_SET[0]
            location = loc[0]
            for i in range(len(velocity_states_final)):
                if velocity_states_final[i] == 'I':
                    color = color_set[0]
                elif velocity_states_final[i] == 'D':
                    color = color_set[1]
                elif velocity_states_final[i] == 'K':
                    color = color_set[2]
                # elif velocity_states_final[i] == 'D':
                #     color = color_set[3]
                else:
                    color=[]
                if i < len(velocity_states_final)-1:
                    ax.fill_between([velocity_turning_points_final[i],velocity_turning_points_final[i+1]],location-0.5,location+0.5,color=color,alpha=0.8,linewidth=0)
            if axvspan != 0:
                for i in four_variable_turning_points:
                    plt.axvline(x=i, color='black', linestyle='-', alpha=1)
            if single_vehicle_data.size !=0:
                data = np.array(single_vehicle_data['smooth_v_follow'].values)
                normalized_data =  (data - np.min(data)) / (np.max(data) - np.min(data))
                plt.plot(single_vehicle_data['FrameID'].values, normalized_data+location-0.5, color = 'red', linestyle='-', linewidth=1)
        if acceleration:
            color_set = COLOR_SET[1]
            location = loc[1]
            for i in range(len(acceleration_states_final)):
                if acceleration_states_final[i] == 'I':
                    color = color_set[0]
                elif acceleration_states_final[i] == 'D':
                    color = color_set[1]
                elif acceleration_states_final[i] == 'K':
                    color = color_set[2]
                # elif acceleration_states_final[i] == 'D':
                #     color = color_set[3]
                else:
                    color=[]
                if i < len(acceleration_states_final)-1:
                    # print(color,'a')
                    ax.fill_between([acceleration_turning_points_final[i],acceleration_turning_points_final[i+1]],location-0.5,location+0.5,color=color,alpha=0.8,linewidth=0)
            if axvspan != 0:
                for i in four_variable_turning_points:
                    plt.axvline(x=i, color='black', linestyle='-', alpha=1)
            if single_vehicle_data.size !=0:
                data = np.array(single_vehicle_data['a_follow'].values)
                normalized_data =  (data - np.min(data)) / (np.max(data) - np.min(data))
                plt.plot(single_vehicle_data['FrameID'].values, normalized_data+location-0.5, color = 'red', linestyle='-', linewidth=1)
        if distance:
            color_set = COLOR_SET[2]
            location = loc[2]
            for i in range(len(distance_states_final)):
                if distance_states_final[i] == 'I':
                    color = color_set[0]
                elif distance_states_final[i] == 'D':
                    color = color_set[1]
                elif distance_states_final[i] == 'K':
                    color = color_set[2]
                # elif distance_states_final[i] == 'D':
                #     color = color_set[3]
                else:
                    color=[]
                if i < len(distance_states_final)-1:
                    # print(color,'d')
                    ax.fill_between([distance_turning_points_final[i],distance_turning_points_final[i+1]],location-0.5,location+0.5,color=color,alpha=0.8,linewidth=0)
            if axvspan != 0:
                for i in four_variable_turning_points:
                    plt.axvline(x=i, color='black', linestyle='-', alpha=1)
            if single_vehicle_data.size !=0:
                data = np.array(single_vehicle_data['time_headway'].values)
                normalized_data =  (data - np.min(data)) / (np.max(data) - np.min(data))
                plt.plot(single_vehicle_data['FrameID'].values, normalized_data+location-0.5, color = 'red', linestyle='-', linewidth=1)
        if delta_v:
            color_set = COLOR_SET[3]
            location = loc[3]
            for i in range(len(delta_v_states_final)):
                if delta_v_states_final[i] == 'I':
                    color = color_set[0]
                elif delta_v_states_final[i] == 'D':
                    color = color_set[1]
                elif delta_v_states_final[i] == 'K':
                    color = color_set[2]
                # elif delta_v_states_final[i] == 'D':
                #     color = color_set[3]
                else:
                    color=[]
                if i < len(delta_v_states_final)-1:
                    # print(color,'dv')
                    ax.fill_between([delta_v_turning_points_final[i],delta_v_turning_points_final[i+1]],location-0.5,location+0.5,color=color,alpha=0.8,linewidth=0)
            if axvspan != 0:
                for i in four_variable_turning_points:
                    plt.axvline(x=i, color='black', linestyle='-', alpha=1)
            if single_vehicle_data.size !=0:
                data = np.array(single_vehicle_data['smooth_delta_v'].values)
                normalized_data =  (data - np.min(data)) / (np.max(data) - np.min(data))
                plt.plot(single_vehicle_data['FrameID'].values, normalized_data+location-0.5, color = 'red', linestyle='-', linewidth=1)
        y_ticks_loc = [0.5,1.5,2.5,3.5]
        y_ticks = [r'$\Delta v$', r'$d$', r'$a$', r'$v$']
        ax.set_yticks(y_ticks_loc)
        ax.set_yticklabels(y_ticks)
        ax.grid(None)
        plt.xlim(velocity_turning_points_final[0],velocity_turning_points_final[-1])
        ax.set_xlabel('FrameID')
        ax.set_ylabel('Variables')
        # ax.set_title('Driving behavior map (Example of ID245)', fontsize=16)
        plt.show()
        return


In [ ]:
four_df = {}
df = {}
four_turning_points = {}
all_vehicle_df = {}
total_states = []
total_states1 = []
total_states2 = []
mse_results = []
df_action_phase = {}
action_phases_list = []
action_phases_labels_list = []
save_data = {'VehicleID': [], 'Label': [], 'Point': []}
df_label = pd.DataFrame(save_data)
unique_vehicle_ids = all_data['Vehicle_ID'].unique()
for index, desired_vehicle_id in enumerate(unique_vehicle_ids):
    desired_vehicle_id = 270
    print(index)
    sg = Segmentation(all_data, desired_vehicle_id)

    result_dataset_velocity = sg.generate_result_dataset('smooth_v_follow')
    result_dataset_acceleration = sg.generate_result_dataset('a_follow')
    result_dataset_distance = sg.generate_result_dataset('time_headway')
    result_dataset_delta_v = sg.generate_result_dataset('smooth_delta_v')

    theta1_velocity = 1.5  # 101
    theta2_velocity = -theta1_velocity
    theta1_acceleration = 0.25
    theta2_acceleration = -theta1_acceleration
    theta1_distance = 0.15
    theta2_distance = -theta1_distance
    theta1_delta_v = 0.8
    theta2_delta_v = -theta1_delta_v

    gama = 20
    single_vehicle_data, delta_t_v, delta_y_v, time_turning_points_v, turning_points_values_v = sg.find_data(result_dataset_velocity)
    _, delta_t_a, delta_y_a, time_turning_points_a, turning_points_values_a = sg.find_data(result_dataset_acceleration)
    _, delta_t_d, delta_y_d, time_turning_points_d, turning_points_values_d = sg.find_data(result_dataset_distance)
    _, delta_t_dv, delta_y_dv, time_turning_points_dv, turning_points_values_dv = sg.find_data(result_dataset_delta_v)

    states_v = sg.get_states_set(delta_y_v, theta1_velocity, theta2_velocity)
    states_a = sg.get_states_set(delta_y_a, theta1_acceleration, theta2_acceleration)
    states_d = sg.get_states_set(delta_y_d, theta1_distance, theta2_distance)
    states_dv = sg.get_states_set(delta_y_dv, theta1_delta_v, theta2_delta_v)

    turning_points_final_v, merged_index_set_v = sg.merge(states_v, delta_t_v, delta_y_v, time_turning_points_v,
                                                          theta1_velocity, theta2_velocity, gama)
    turning_points_final_a, merged_index_set_a = sg.merge(states_a, delta_t_a, delta_y_a, time_turning_points_a,
                                                          theta1_acceleration, theta2_acceleration, gama)
    turning_points_final_d, merged_index_set_d = sg.merge(states_d, delta_t_d, delta_y_d, time_turning_points_d,
                                                          theta1_distance, theta2_distance, gama)
    turning_points_final_dv, merged_index_set_dv = sg.merge(states_dv, delta_t_dv, delta_y_dv,
                                                            time_turning_points_dv, theta1_delta_v, theta2_delta_v,
                                                            gama)

    states_v_final = sg.cal_states_final(states_v, 'smooth_v_follow', time_turning_points_v, merged_index_set_v)
    states_a_final = sg.cal_states_final(states_a, 'a_follow', time_turning_points_a, merged_index_set_a)
    states_d_final = sg.cal_states_final(states_d, 'time_headway', time_turning_points_d, merged_index_set_d)
    states_dv_final = sg.cal_states_final(states_dv, 'smooth_delta_v', time_turning_points_dv, merged_index_set_dv)

    four_variable_turning_points, _ = sg.total_turning_points(turning_points_final_v, turning_points_final_a,
                                                              turning_points_final_d, turning_points_final_dv, gama)

    four_variable_turning_points_wo_gama = sg.total_turning_points_without_gamma(turning_points_final_v,
                                                                                 turning_points_final_a,
                                                                                 turning_points_final_d,
                                                                                 turning_points_final_dv)
    # print(four_variable_turning_points_wo_gama,'eeeeeeeeeeeee')
    v_final_point4, v_final_states4 = sg.four_variable_point_state(four_variable_turning_points_wo_gama,
                                                                   turning_points_final_v, states_v_final)
    a_final_point4, a_final_states4 = sg.four_variable_point_state(four_variable_turning_points_wo_gama,
                                                                   turning_points_final_a, states_a_final)
    d_final_point4, d_final_states4 = sg.four_variable_point_state(four_variable_turning_points_wo_gama,
                                                                   turning_points_final_d, states_d_final)
    dv_final_point4, dv_final_states4 = sg.four_variable_point_state(four_variable_turning_points_wo_gama,
                                                                     turning_points_final_dv, states_dv_final)
    # print(v_final_point4,v_final_states4,a_final_point4,a_final_states4 ,d_final_point4,d_final_states4 ,'eeeeeeeeeerrrrrrrrrrr')

    v_final_point4, v_final_states4 = sg.merge_adj_same_states(v_final_point4, v_final_states4)
    a_final_point4, a_final_states4 = sg.merge_adj_same_states(a_final_point4, a_final_states4)
    d_final_point4, d_final_states4 = sg.merge_adj_same_states(d_final_point4, d_final_states4)
    dv_final_point4, dv_final_states4 = sg.merge_adj_same_states(dv_final_point4, dv_final_states4)
    # four_variable_turning_points,four_real_points = sg.total_turning_points(v_final_point4,a_final_point4,d_final_point4,dv_final_point4,gama)
    four_variable_turning_points_wo_gama = sg.total_turning_points_without_gamma(v_final_point4, a_final_point4,
                                                                                 d_final_point4, dv_final_point4)
    four_real_points = four_variable_turning_points_wo_gama

    v_all_state4 = sg.all_states(four_variable_turning_points_wo_gama, v_final_point4, v_final_states4)
    a_all_state4 = sg.all_states(four_variable_turning_points_wo_gama, a_final_point4, a_final_states4)
    d_all_state4 = sg.all_states(four_variable_turning_points_wo_gama, d_final_point4, d_final_states4)
    dv_all_state4 = sg.all_states(four_variable_turning_points_wo_gama, dv_final_point4, dv_final_states4)

    # binary_state,binary_state1,binary_state2 = sg.binary_states(four_variable_turning_points_wo_gama,v_all_state4,a_all_state4,d_all_state4,dv_all_state4,gama)

    four_variable_turning_points_wo_gama, binary_state, binary_state1, binary_state2 = sg.binary_states_del_too_short(
        four_variable_turning_points_wo_gama, v_all_state4, a_all_state4, d_all_state4, dv_all_state4, gama)
    # print(binary_state,'llllooo')
    total_states.append(binary_state)
    total_states1.append(binary_state1)
    total_states2.append(binary_state2)

    # segment_points, action_phase_selection = sg.choose_action_phase(four_variable_turning_points_wo_gama, single_vehicle_data)
    segment_points, action_phase_selection, label_set_selection = sg.choose_action_phase(four_variable_turning_points_wo_gama, single_vehicle_data, binary_state1)

    four_df['velocity_turning_points'] = v_final_point4
    four_df['velocity_states'] = v_final_states4
    four_df['acceleration_turning_points'] = a_final_point4
    four_df['acceleration_states'] = a_final_states4
    four_df['distance_turning_points'] = d_final_point4
    four_df['distance_states'] = d_final_states4
    four_df['delta_v_turning_points'] = dv_final_point4
    four_df['delta_v_states'] = dv_final_states4
    four_df['four_turning_points'] = four_variable_turning_points
    four_df['four_real_points'] = four_real_points

    df['velocity_turning_points'] = turning_points_final_v
    df['velocity_states'] = states_v_final
    df['acceleration_turning_points'] = turning_points_final_a
    df['acceleration_states'] = states_a_final
    df['distance_turning_points'] = turning_points_final_d
    df['distance_states'] = states_d_final
    df['delta_v_turning_points'] = turning_points_final_dv
    df['delta_v_states'] = states_dv_final
    all_vehicle_df['{0}'.format(desired_vehicle_id)] = df
    df_action_phase['{0}'.format(desired_vehicle_id)] = [action_phase_selection, binary_state[: -1]]
    action_phases_list.append(action_phase_selection)
    action_phases_labels_list.append(label_set_selection)

    rule_based_label, rule_based_points = sg.rule_label(states_v_final, turning_points_final_v, states_a_final,
                                                        turning_points_final_a, states_d_final,
                                                        turning_points_final_d)
    four_states = sg.find_four_states(four_df)

    rule_based_label = []
    rule_based_points = []
    for v_index, state_v in enumerate(states_v_final):
        if state_v == 'I':
            rule_based_label.append('Speed_up')
            rule_based_points.append(turning_points_final_v[v_index])
        elif state_v == 'D':
            rule_based_label.append('Slow_down')
            rule_based_points.append(turning_points_final_v[v_index])
        else:
            turning_point_v = turning_points_final_v[v_index]
            d_ = turning_points_final_d[turning_points_final_d <= turning_point_v].max() if np.any(turning_points_final_d <= turning_point_v) else None
            index_d_ = np.where(turning_points_final_d == d_)[0][0] if d_ is not None else None
            if turning_points_final_d[index_d_] in rule_based_points:
                continue
            if len(rule_based_points) != 0:
                if turning_points_final_d[index_d_] - rule_based_points[-1] < 10:
                    continue
            if states_d_final[index_d_] == 'I':
                rule_based_label.append('Fall_behind')
                rule_based_points.append(turning_points_final_d[index_d_])
            elif states_d_final[index_d_] == 'D':
                rule_based_label.append('Catch_up') 
                rule_based_points.append(turning_points_final_d[index_d_])
            else:
                turning_point_d = turning_points_final_d[index_d_]
                a_ = turning_points_final_a[turning_points_final_a <= turning_point_d].max() if np.any(turning_points_final_a <= turning_point_d) else None
                index_a_ = np.where(turning_points_final_a == a_)[0][0] if a_ is not None else None
                if turning_points_final_a[index_a_] in rule_based_points:
                    continue
                if len(rule_based_points) != 0:
                    if turning_points_final_a[index_a_] - rule_based_points[-1] < 10:
                        continue
                if states_a_final[index_a_] == 'I':
                    rule_based_label.append('Follow_behind')
                    rule_based_points.append(turning_points_final_a[index_a_])
                elif states_a_final[index_a_] == 'D':
                    rule_based_label.append('Follow_behind')
                    rule_based_points.append(turning_points_final_a[index_a_])
                else:
                    rule_based_label.append('Hold_speed')
                    rule_based_points.append(turning_points_final_a[index_a_])
    
    # for label, point in zip(rule_based_label, rule_based_points):
    df_label = df_label.append({'VehicleID': desired_vehicle_id, 'Label': rule_based_label, 'Point': rule_based_points}, ignore_index=True)
    
    sg.plt_segment('smooth_v_follow','Velocity',single_vehicle_data,states_v_final,turning_points_final_v,time_turning_points_v)
    sg.plt_segment('a_follow','Acceleration',single_vehicle_data,states_a_final,turning_points_final_a,time_turning_points_a)
    sg.plt_segment('time_headway','Distance',single_vehicle_data,states_d_final,turning_points_final_d,time_turning_points_d)
    sg.plt_segment('smooth_delta_v','Delta_v',single_vehicle_data,states_dv_final,turning_points_final_dv,time_turning_points_dv)
    if index > 20:
        sg.plt_rule_label(single_vehicle_data,rule_based_label,rule_based_points)
    sg.fill_color(1,1,1,1,dataframe = df,axvspan=0,four_variable_turning_points=[],single_vehicle_data=np.empty((0,0)))

    sg.fill_color(1,1,1,1,four_df,1,four_variable_turning_points_wo_gama,single_vehicle_data)
    sg.plt_rule_label(single_vehicle_data,rule_based_label,rule_based_points)
    if index > 10:
        break

df_label.to_csv("labeled_data.csv", index=False)

In [238]:
rule_based_label = []
rule_based_points = []
for v_index, state_v in enumerate(states_v_final):
    if state_v == 'I':
        rule_based_label.append('Speed_up')
        rule_based_points.append(turning_points_final_v[v_index])
    elif state_v == 'D':
        rule_based_label.append('Slow_down')
        rule_based_points.append(turning_points_final_v[v_index])
    else:
        turning_point_v = turning_points_final_v[v_index]
        d_ = turning_points_final_d[turning_points_final_d <= turning_point_v].max() if np.any(turning_points_final_d <= turning_point_v) else None
        index_d_ = np.where(turning_points_final_d == d_)[0][0] if d_ is not None else None
        if turning_points_final_d[index_d_] in rule_based_points:
            continue
        if len(rule_based_points) != 0:
            if turning_points_final_d[index_d_] - rule_based_points[-1] < 10:
                continue
        if states_d_final[index_d_] == 'I':
            rule_based_label.append('Catch_up')
            rule_based_points.append(turning_points_final_d[index_d_])
        elif states_d_final[index_d_] == 'D':
            rule_based_label.append('Fall_behind') 
            rule_based_points.append(turning_points_final_d[index_d_])
        else:
            turning_point_d = turning_points_final_d[index_d_]
            a_ = turning_points_final_a[turning_points_final_a <= turning_point_d].max() if np.any(turning_points_final_a <= turning_point_d) else None
            index_a_ = np.where(turning_points_final_a == a_)[0][0] if a_ is not None else None
            if turning_points_final_a[index_a_] in rule_based_points:
                continue
            if len(rule_based_points) != 0:
                if turning_points_final_a[index_a_] - rule_based_points[-1] < 10:
                    continue
            if states_a_final[index_a_] == 'I':
                rule_based_label.append('Follow_behind')
                rule_based_points.append(turning_points_final_a[index_a_])
            elif states_a_final[index_a_] == 'D':
                rule_based_label.append('Follow_behind')
                rule_based_points.append(turning_points_final_a[index_a_])
            else:
                rule_based_label.append('Hold_speed')
                rule_based_points.append(turning_points_final_a[index_a_])

In [264]:
rule_based_points

[0, 75, 157]

In [265]:
rule_based_label

['Fall_behind', 'Fall_behind', 'Fall_behind']

In [ ]:
import ast

all_data = pd.read_csv('trainHH/trainHH_data.csv')
num_vehicles = all_data['Vehicle_ID'].nunique()
label_info = df_label 

def segment_trajectory(vehicle_id, all_data, turning_points):
    segment = []
    df = all_data[all_data['Vehicle_ID'] == vehicle_id]
    for i in range(len(turning_points) - 1):
        start = turning_points[i]
        end = turning_points[i + 1]
        segment = df.iloc[start:end][['smooth_v_follow', 'a_follow', 'smooth_delta_v', 'time_headway']]
        segment_array = segment.to_numpy().T  # Transpose to get each variable as a separate list
        combined_segments.append(segment_array.tolist())
    return combined_segments

def get_points_and_labels(vehicle_id, label_info):
    vehicle_data = label_info[label_info['VehicleID'] == vehicle_id]
    if not vehicle_data.empty:
        points_string = vehicle_data['Point'].iloc[0]
        labels_string = vehicle_data['Label'].iloc[0]
        # Safely evaluate the string as a list
        points = ast.literal_eval(points_string)
        labels = ast.literal_eval(labels_string)
        return points, labels[:-1]  # Exclude the last label
    else:
        return [], []

combined_segments = []
all_labels = []
for index in label_info['VehicleID']:
    vehicle_id = index
    turning_points, labels = get_points_and_labels(index, label_info)
    all_labels.extend(labels)
    combined_segments = segment_trajectory(vehicle_id, all_data, turning_points)

In [ ]:
# read labels
file_path = 'labeled_data/labels.txt'
labels = []
with open(file_path, 'r') as file:
    for line in file:
        label = line.strip()
        labels.append(label)

Catch_up


In [ ]:
# save labeled data
import json

# Save as JSON
with open('labeled_data.json', 'w') as json_file:
    json.dump(combined_segments, json_file)